In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/rotulado_v1.csv").dropna(subset=["categoria"])

df["descricao"].str.split(" - ").str[0].value_counts().head(15)


descricao
Compra no débito                   204
Transferência enviada pelo Pix      25
Pagamento de boleto efetuado        18
Pagamento efetuado                  16
Compra Meio De Transporte            9
Aplicação                            8
Cashback                             7
Transferência recebida pelo Pix      6
Resgate                              5
Estorno                              2
Pix recebido                         1
Pix enviado                          1
Pix enviado devolvido                1
Crédito liberado                     1
NETFLIX.COM                          1
Name: count, dtype: int64

In [2]:
df["descricao"].str.split().str[-1].value_counts().head(15)

descricao
Bra         204
LTDA         11
Sa            8
PESSOA        6
S.A.          5
01/03         5
-CT           5
A             4
01/02         4
02/06         4
R             3
BERNARDO      3
SBC           3
01/06         3
(0077)        2
Name: count, dtype: int64

In [3]:
import re

t = re.sub(r"([a-z])(\d)", r"\1 \2", "drogasil2705")

re.sub(r"\d+", "", t)

'drogasil '

In [4]:
re.sub(r"\s+bra$", "", "sao paulo bra")

'sao paulo'

In [5]:
re.sub(r"no estabelecimento\s*+\s*", " ", "no estabelecimento marmel mf indaiatuba")

' marmel mf indaiatuba'

In [6]:
re.sub(r"\d{2}/\d{2}", " ", "01/03")

' '

In [7]:
re.sub(r"\d+ de \d+", " ", "05 de 10")

' '

In [8]:
re.sub(r"-ct$", "", "Basquete-ct")

'Basquete'

In [9]:
re.sub(r"[^\w\s]", " ", "ifd*pizzaria - netflix.com (sp)")

'ifd pizzaria   netflix com  sp '

In [10]:
import sys
sys.path.append("..")            
from src.limpeza import limpar

exemplos = [
    "Compra no débito - Drogasil2705 Sao Bernardo Bra",
    "Compra no débito - No Estabelecimento Marmel Mf Indaiatuba Bra",
    "LATAM AIR*0000 01/03",
    "ALLIANZ SEGU*04 de 10",
    "FORT 585 SP -CT",
    "Pix enviado - PESSOA - NU PAGAMENTOS - IP",
]
for e in exemplos:
    print(f"{e!r:65} -> {limpar(e)!r}")

'Compra no débito - Drogasil2705 Sao Bernardo Bra'                -> 'drogasil sao bernardo'
'Compra no débito - No Estabelecimento Marmel Mf Indaiatuba Bra'  -> 'marmel mf indaiatuba'
'LATAM AIR*0000 01/03'                                            -> 'latam air'
'ALLIANZ SEGU*04 de 10'                                           -> 'allianz segu'
'FORT 585 SP -CT'                                                 -> 'fort sp'
'Pix enviado - PESSOA - NU PAGAMENTOS - IP'                       -> 'pix enviado pessoa nu pagamentos ip'


In [11]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

X, y = df["descricao"], df["categoria"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # o MESMO cv do baseline

baseline = make_pipeline(
    TfidfVectorizer(lowercase=True),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)
com_limpeza = make_pipeline(
    TfidfVectorizer(preprocessor=limpar),
    LogisticRegression(max_iter=1000, class_weight="balanced"),
)

for nome, modelo in [("baseline", baseline), ("com limpeza", com_limpeza)]:
    f1 = cross_val_score(modelo, X, y, cv=cv, scoring="f1_macro")
    acc = cross_val_score(modelo, X, y, cv=cv, scoring="accuracy")
    print(f"{nome:12} F1 macro {f1.mean():.3f} ± {f1.std():.3f} | acurácia {acc.mean():.3f} ± {acc.std():.3f}")

baseline     F1 macro 0.631 ± 0.028 | acurácia 0.620 ± 0.030
com limpeza  F1 macro 0.672 ± 0.028 | acurácia 0.657 ± 0.027


In [12]:
from sklearn.model_selection import learning_curve

learning_curve(com_limpeza, X, y, cv=cv, scoring="f1_macro", train_sizes=[0.2, 0.4, 0.6, 0.8, 1.0])

(array([119, 238, 357, 476, 595]),
 array([[0.98191315, 1.        , 1.        , 0.99331958, 0.97647214],
        [0.97698393, 0.97317347, 0.98097021, 0.97689851, 0.99036505],
        [0.97141394, 0.96096809, 0.97179849, 0.97394176, 0.97196307],
        [0.97108654, 0.9639246 , 0.95842425, 0.97329151, 0.97348272],
        [0.94538745, 0.93781033, 0.95172942, 0.94886765, 0.95487131]]),
 array([[0.34514808, 0.35586623, 0.27689448, 0.23282931, 0.33110655],
        [0.41981526, 0.36710814, 0.31982006, 0.34301495, 0.41755586],
        [0.56320363, 0.52317778, 0.44778757, 0.41873414, 0.51564981],
        [0.58796399, 0.56967035, 0.49968813, 0.47412102, 0.53043929],
        [0.6763197 , 0.68743754, 0.68717353, 0.61654742, 0.69038621]]))